<a href="https://colab.research.google.com/github/hsmu-jeongeun/medical-data-analysis/blob/main/07_Tensorflow_using_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 2 : GPU의 이해

- TensorFlow가 내 컴퓨터의 GPU를 정상적으로 인식하는지 확인하고, 거대 행렬 곱셈 시 CPU와 GPU의 연산 속도 차이를 직접 측정해보기

## 라이브러리 불러오기

In [1]:
import tensorflow as tf
import time
import numpy as np

## GPU 확인하기

In [2]:
# 1. TensorFlow 버전 확인
print(f"TensorFlow 버전: {tf.__version__}")

# 2. GPU 인식 확인
physical_devices = tf.config.list_physical_devices('GPU')
print(f"사용 가능한 GPU 개수:{len(physical_devices)}")

if len(physical_devices) > 0:
    print("Available GPU devices")
    for device in physical_devices:
        print(f"장치 세부 정보: {device}")
else:
    print("Cannot find GPU device.")

TensorFlow 버전: 2.19.0
사용 가능한 GPU 개수:1
Available GPU devices
장치 세부 정보: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


## CPU vs GPU 연산 속도 비교
- CPU: 복잡한 단일 연산에 유리
- GPU: 단순 반복 행렬 연산에 압도적으로 유리

### 거대 행렬 곱셈 테스트
- 딥러닝의 가중치 업데이트는 큰 행렬들을 곱하고 더하는 과정
- 실험 방법
  - $10,000 \times 10,000$ 크기의 거대 난수 행렬 두 개 생성
  - 이를 곱할 때 순수 CPU만 사용할 때와 GPU를 강제로 할당하여 사용할 때의 걸리는 시간 측정 및 비교

In [3]:
size = 10000 # 5000으로 변경 가능

- CPU 연산 속도 측정

In [4]:
# tf.device('/CPU:0') : 강제 CPU 연산 지시
with tf.device('/CPU:0'):
    # 랜덤 행렬 생성
    matrix_a_cpu = tf.random.normal([size, size])
    matrix_b_cpu = tf.random.normal([size, size])

    start_time = time.time()
    # 행렬 곱셈 연산 : tf.matmul
    result_cpu = tf.matmul(matrix_a_cpu, matrix_b_cpu)
    cpu_time = time.time() - start_time

print(f"CPU 연산 소요 시간: {cpu_time:.4f} 초")

CPU 연산 소요 시간: 18.8977 초


- GPU 연산 속도 측정

In [5]:
if len(physical_devices) > 0:
    # tf.device('/GPU:0'): 첫 번째 GPU 연산 지시
    with tf.device('/GPU:0'):
        # 랜덤 행렬 생성
        matrix_a_gpu = tf.random.normal([size, size])
        matrix_b_gpu = tf.random.normal([size, size])

        # GPU는 처음 실행될 때 메모리 초기화 등으로 시간이 걸릴 수 있으므로
        # Dummy 연산을 한 번 수행하는 것이 정확한 측정에 좋음
        _ = tf.matmul(matrix_a_gpu, matrix_b_gpu)

        start_time = time.time()
        # 실제 행렬 곱셈 연산
        result_gpu = tf.matmul(matrix_a_gpu, matrix_b_gpu)
        gpu_time = time.time() - start_time

    print(f"GPU 연산 소요 시간: {gpu_time:.4f} 초")
    print(f"결과 : GPU가 CPU보다 약 {cpu_time / gpu_time:.1f}배 빠르다")
else:
    print("\nGPU가 없습니다")

GPU 연산 소요 시간: 0.0003 초
결과 : GPU가 CPU보다 약 57105.7배 빠르다
